# Hyperparameter Tuning 


The goal is to find a set of quasi-optimal hyperparameters to improve on the model's performance on the test data set. We will be using cross validation on the train dataset.

### Challenge of balancing between finding the optimal combination of Features/ Model and preventing number of combinations

- **Sequentially Tuning (Features then Model)** is more computationally manageable, but risk finding a set of features that is only optimal for a sub-optimal model (Local Maxima). The true global maxima might require a different set of features with a well-tuned model.
- **Concurrent Tuning (Features and Model in the same search space)** Can find the local maxima set of features and model hyperparameters, but it can require an enormous amount of trials, making it computationally expensive. 

### Solution

We will be adopting a sequential, iterative approach. This is based on the assumption that good features will be useful for both a default model and well tuned model. 

#### Stage 1:
For the first stage of finding a good-enough set of feature hyperparameters, we will be using an untuned LightGBM as the default model for tree-based models. LightGBM handles raw values well, and is powerful enough to give strong performance signal. For linear regression model, we will use untuned logistic regression itself.

#### Stage 2:
Following which, we freeze the features from first stage and use this parameter to tune our set of models.


In [ ]:
# Imports
import numpy as np
import pandas as pd
from typing import List, Dict, Any

from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

import lightgbm as lgb
import optuna, optuna_dashboard

optuna.logging.set_verbosity(optuna.logging.WARNING)


In [ ]:
from lib.data_management import load_match_tables_jsonl

pro_match_file_path = "../data/pro_match_tables.jsonl"
match_list = load_match_tables_jsonl(pro_match_file_path)
len(match_list)


In [ ]:
# Defensive sorting of matches by start_time
sorted_match_list = sorted(match_list, key=lambda x: x.start_time)
len(sorted_match_list)

In [ ]:
match_outcome_list = [match.outcome for match in sorted_match_list]

In [ ]:
train_percentage = 0.8
n_train = int(len(sorted_match_list) * train_percentage)

train_match, test_match = sorted_match_list[:n_train], sorted_match_list[n_train:]
len(train_match), len(test_match)

In [ ]:
train_outcome = [match.outcome for match in train_match]
test_outcome = [match.outcome for match in test_match]

In [ ]:
len(train_match), len(test_match)

In [ ]:
outcome_df = pd.DataFrame(match.model_dump() for match in match_outcome_list)

### Creating the Hold out set (Same as test_set in untuned version)

In [ ]:
outcome_df.sort_values("start_time", inplace=True)

## Feature Tuning

In [ ]:
from dota_oracle_pipeline.feature_engineering.batch import TeamDecayFeatureGenerator, PlayerHeroDynamicPriorFeatureGenerator, HeroWinrateDecayFeatureGenerator
from lib.utils import merge_features_on_match_id

In [ ]:
team_decay_generator = TeamDecayFeatureGenerator()
hero_winrate_decay_generator = HeroWinrateDecayFeatureGenerator()
player_hero_dynamic_prior_generator = PlayerHeroDynamicPriorFeatureGenerator()

In [ ]:
SEARCH_SPACE_CONFIG = {
    "team_decay_alpha":           {"type": "int", "low": 1, "high": 100},
    "team_decay_beta":            {"type": "int", "low": 2, "high": 1000},
    "team_decay_half_life_days":  {"type": "categorical", "choices": [7, 14, 30, 60, 90]},

    # Hero Winrate Decay Parameters
    "hero_winrate_decay_alpha":          {"type": "float", "low": 0.5, "high": 20.0, "log": True},
    "hero_winrate_decay_beta":           {"type": "float", "low": 5.0, "high": 50.0, "log": True},
    "hero_winrate_decay_half_life_days": {"type": "categorical", "choices": [7, 14, 30, 60, 90]},

    # Player Hero Dynamic Prior Parameters
    "phdp_player_credibility_C":   {"type": "int",   "low": 5, "high": 50},
    "phdp_player_half_life_days":  {"type": "float", "low": 15.0, "high": 120.0},
    "phdp_hero_alpha":             {"type": "float", "low": 0.5, "high": 20.0, "log": True},
    "phdp_hero_beta":              {"type": "float", "low": 5.0, "high": 50.0, "log": True},
    "phdp_hero_half_life_days":    {"type": "float", "low": 30.0, "high": 180.0},
}

In [ ]:
def objective(trial: optuna.trial.Trial) -> float:

    team_decay_alpha = trial.suggest_int("team_decay_alpha", 1, 100)
    team_decay_beta = trial.suggest_int("team_decay_beta", 2, 1000)
    team_decay_half_life_days = trial.suggest_categorical("team_decay_half_life_days", [7, 14, 30, 60, 90])

    hero_winrate_decay_alpha = trial.suggest_int("hero_winrate_decay_alpha", 1, 100)
    hero_winrate_decay_beta = trial.suggest_int("hero_winrate_decay_beta", 2, 1000)
    hero_winrate_decay_half_life_days = trial.suggest_categorical("hero_winrate_decay_half_life_days", [7, 14, 30, 60, 90])
    
    phdp_player_credibility_C = trial.suggest_int("phdp_player_credibility_C", 5, 50)
    phdp_player_half_life_days = trial.suggest_int("phdp_player_half_life_days", 15, 120)
    phdp_hero_alpha = trial.suggest_int("phdp_hero_alpha", 1, 100)
    phdp_hero_beta = trial.suggest_int("phdp_hero_beta", 2, 1000)
    phdp_hero_half_life_days = trial.suggest_categorical("phdp_hero_half_life_days", [30, 60, 90, 120, 180])

    # 2. Generate Features
    
    team_decay_features = team_decay_generator.generate(
        match_list,
        alpha=team_decay_alpha,
        beta=team_decay_beta,
        half_life_days=team_decay_half_life_days
    )
    
    hero_winrate_decay_features = hero_winrate_decay_generator.generate(
        match_list,
        alpha=hero_winrate_decay_alpha,
        beta=hero_winrate_decay_beta,
        half_life_days=hero_winrate_decay_half_life_days
    )
    
    player_hero_dynamic_prior_features = player_hero_dynamic_prior_generator.generate(
        match_list,
        player_credibility_C=phdp_player_credibility_C,
        player_half_life_days=phdp_player_half_life_days,
        hero_alpha=phdp_hero_alpha,
        hero_beta=phdp_hero_beta,
        hero_half_life_days=phdp_hero_half_life_days
    )
    
    
    team_decay_df = pd.DataFrame(instance.model_dump() for instance in team_decay_features)
    hero_winrate_decay_df = pd.DataFrame(instance.model_dump() for instance in hero_winrate_decay_features)
    player_hero_dynamic_prior_df = pd.DataFrame(instance.model_dump() for instance in player_hero_dynamic_prior_features)

    combined_df = merge_features_on_match_id([outcome_df, team_decay_df, hero_winrate_decay_df, player_hero_dynamic_prior_df])

    y = combined_df['radiant_win']
    X = combined_df.drop(columns=['match_id', 'radiant_win'])
    
    model = lgb.LGBMClassifier(objective='binary', n_estimators=100, random_state=42, verbose=-1)
    
    time_series_cv = TimeSeriesSplit(n_splits=3)
    
    scores = cross_val_score(model, X, y, cv=time_series_cv, scoring='accuracy', n_jobs=-1)
    
    final_score = scores.mean()
    
    return final_score
    

In [ ]:
def objective_stage1(trial: optuna.trial.Trial) -> float:
    # 1. TUNE these parameters
    team_decay_alpha = trial.suggest_int("team_decay_alpha", 1, 100)
    team_decay_beta = trial.suggest_int("team_decay_beta", 2, 1000)
    team_decay_half_life_days = trial.suggest_categorical("team_decay_half_life_days", [7, 14, 30, 60, 90])

    # FIXED: Using reasonable defaults for other feature groups in this stage
    hero_winrate_decay_alpha = 15
    hero_winrate_decay_beta = 15
    hero_winrate_decay_half_life_days = 60
    phdp_player_credibility_C = 25
    phdp_player_half_life_days = 60
    phdp_hero_alpha = 15
    phdp_hero_beta = 15
    phdp_hero_half_life_days = 90

    # 2. Generate Features (using a mix of tuned and fixed params)
    team_decay_features = team_decay_generator.generate(
        match_list, alpha=team_decay_alpha, beta=team_decay_beta, half_life_days=team_decay_half_life_days
    )
    hero_winrate_decay_features = hero_winrate_decay_generator.generate(
        match_list, alpha=hero_winrate_decay_alpha, beta=hero_winrate_decay_beta, half_life_days=hero_winrate_decay_half_life_days
    )
    player_hero_dynamic_prior_features = player_hero_dynamic_prior_generator.generate(
        match_list, player_credibility_C=phdp_player_credibility_C, player_half_life_days=phdp_player_half_life_days,
        hero_alpha=phdp_hero_alpha, hero_beta=phdp_hero_beta, hero_half_life_days=phdp_hero_half_life_days
    )
    
    # 3. Combine and Evaluate (this part remains the same in all functions)
    team_decay_df = pd.DataFrame(instance.model_dump() for instance in team_decay_features)
    hero_winrate_decay_df = pd.DataFrame(instance.model_dump() for instance in hero_winrate_decay_features)
    player_hero_dynamic_prior_df = pd.DataFrame(instance.model_dump() for instance in player_hero_dynamic_prior_features)

    combined_df = merge_features_on_match_id([outcome_df, team_decay_df, hero_winrate_decay_df, player_hero_dynamic_prior_df])

    y = combined_df['radiant_win']
    X = combined_df.drop(columns=['match_id', 'radiant_win'])
    
    model = lgb.LGBMClassifier(objective='binary', n_estimators=100, random_state=42, verbose=-1)
    time_series_cv = TimeSeriesSplit(n_splits=3)
    scores = cross_val_score(model, X, y, cv=time_series_cv, scoring='accuracy', n_jobs=-1)
    
    return scores.mean()

In [ ]:
study_stage1 = optuna.create_study(direction="maximize", study_name="dota_tuning_stage1_team_decay")
study_stage1.optimize(objective_stage1, n_trials=30, callbacks=[cb])
print("Best params for Stage 1:", study_stage1.best_params, study_stage1.best_value)

In [ ]:
def objective_stage2(trial: optuna.trial.Trial) -> float:
    # FIXED: Use the best values from your Stage 1 study!
    team_decay_alpha = 47  # <-- REPLACE with your best alpha from Stage 1
    team_decay_beta = 207 # <-- REPLACE with your best beta from Stage 1
    team_decay_half_life_days = 30 # <-- REPLACE with your best half_life from Stage 1

    # 1. TUNE these parameters
    hero_winrate_decay_alpha = trial.suggest_int("hero_winrate_decay_alpha", 1, 100)
    hero_winrate_decay_beta = trial.suggest_int("hero_winrate_decay_beta", 2, 1000)
    hero_winrate_decay_half_life_days = trial.suggest_categorical("hero_winrate_decay_half_life_days", [7, 14, 30, 60, 90])
    
    # FIXED: Using reasonable defaults for the last feature group
    phdp_player_credibility_C = 25
    phdp_player_half_life_days = 60
    phdp_hero_alpha = 15
    phdp_hero_beta = 15
    phdp_hero_half_life_days = 90

    # 2. Generate Features...
    # (The rest of the function is identical to objective_stage1)
    team_decay_features = team_decay_generator.generate(
        match_list, alpha=team_decay_alpha, beta=team_decay_beta, half_life_days=team_decay_half_life_days
    )
    hero_winrate_decay_features = hero_winrate_decay_generator.generate(
        match_list, alpha=hero_winrate_decay_alpha, beta=hero_winrate_decay_beta, half_life_days=hero_winrate_decay_half_life_days
    )
    player_hero_dynamic_prior_features = player_hero_dynamic_prior_generator.generate(
        match_list, player_credibility_C=phdp_player_credibility_C, player_half_life_days=phdp_player_half_life_days,
        hero_alpha=phdp_hero_alpha, hero_beta=phdp_hero_beta, hero_half_life_days=phdp_hero_half_life_days
    )
    
    # 3. Combine and Evaluate...
    team_decay_df = pd.DataFrame(instance.model_dump() for instance in team_decay_features)
    hero_winrate_decay_df = pd.DataFrame(instance.model_dump() for instance in hero_winrate_decay_features)
    player_hero_dynamic_prior_df = pd.DataFrame(instance.model_dump() for instance in player_hero_dynamic_prior_features)

    combined_df = merge_features_on_match_id([outcome_df, team_decay_df, hero_winrate_decay_df, player_hero_dynamic_prior_df])

    y = combined_df['radiant_win']
    X = combined_df.drop(columns=['match_id', 'radiant_win'])
    
    model = lgb.LGBMClassifier(objective='binary', n_estimators=100, random_state=42, verbose=-1)
    time_series_cv = TimeSeriesSplit(n_splits=3)
    scores = cross_val_score(model, X, y, cv=time_series_cv, scoring='accuracy', n_jobs=-1)
    
    return scores.mean()

In [ ]:
study_stage2 = optuna.create_study(direction="maximize", study_name="dota_tuning_stage2_team_decay")
study_stage2.optimize(objective_stage2, n_trials=40, callbacks=[cb])
print("Best params for Stage 2:", study_stage2.best_params, study_stage2.best_value)

In [ ]:
def objective_stage3(trial: optuna.trial.Trial) -> float:
    # FIXED: Use the best values from your Stage 1 study
    team_decay_alpha = 47  # <-- REPLACE with your best alpha from Stage 1
    team_decay_beta = 207 # <-- REPLACE with your best beta from Stage 1
    team_decay_half_life_days = 30 # <-- REPLACE with your best half_life from Stage 1

    # FIXED: Use the best values from your Stage 2 study
    hero_winrate_decay_alpha = 67 # <-- REPLACE with your best alpha from Stage 2
    hero_winrate_decay_beta = 176 # <-- REPLACE with your best beta from Stage 2
    hero_winrate_decay_half_life_days = 90 # <-- REPLACE with your best half_life from Stage 2
    
    # 1. TUNE these parameters
    phdp_player_credibility_C = trial.suggest_int("phdp_player_credibility_C", 5, 50)
    phdp_player_half_life_days = trial.suggest_int("phdp_player_half_life_days", 15, 120)
    phdp_hero_alpha = trial.suggest_int("phdp_hero_alpha", 1, 100)
    phdp_hero_beta = trial.suggest_int("phdp_hero_beta", 2, 1000)
    phdp_hero_half_life_days = trial.suggest_categorical("phdp_hero_half_life_days", [30, 60, 90, 120, 180])

    # 2. Generate Features...
    # (The rest of the function is identical)
    team_decay_features = team_decay_generator.generate(
        match_list, alpha=team_decay_alpha, beta=team_decay_beta, half_life_days=team_decay_half_life_days
    )
    hero_winrate_decay_features = hero_winrate_decay_generator.generate(
        match_list, alpha=hero_winrate_decay_alpha, beta=hero_winrate_decay_beta, half_life_days=hero_winrate_decay_half_life_days
    )
    player_hero_dynamic_prior_features = player_hero_dynamic_prior_generator.generate(
        match_list, player_credibility_C=phdp_player_credibility_C, player_half_life_days=phdp_player_half_life_days,
        hero_alpha=phdp_hero_alpha, hero_beta=phdp_hero_beta, hero_half_life_days=phdp_hero_half_life_days
    )
    
    # 3. Combine and Evaluate...
    team_decay_df = pd.DataFrame(instance.model_dump() for instance in team_decay_features)
    hero_winrate_decay_df = pd.DataFrame(instance.model_dump() for instance in hero_winrate_decay_features)
    player_hero_dynamic_prior_df = pd.DataFrame(instance.model_dump() for instance in player_hero_dynamic_prior_features)

    combined_df = merge_features_on_match_id([outcome_df, team_decay_df, hero_winrate_decay_df, player_hero_dynamic_prior_df])

    y = combined_df['radiant_win']
    X = combined_df.drop(columns=['match_id', 'radiant_win'])
    
    model = lgb.LGBMClassifier(objective='binary', n_estimators=100, random_state=42, verbose=-1)
    time_series_cv = TimeSeriesSplit(n_splits=3)
    scores = cross_val_score(model, X, y, cv=time_series_cv, scoring='accuracy', n_jobs=-1)
    
    return scores.mean()

In [ ]:
study_stage3 = optuna.create_study(direction="maximize", study_name="dota_tuning_stage3_team_decay")
study_stage3.optimize(objective_stage3, n_trials=100, callbacks=[cb])
print("Best params for Stage 3:", study_stage3.best_params, study_stage3.best_value)

In [ ]:
STUDY_NAME = "team_decay_tuning_brier_score"      
DB_FILENAME = "../data/team_decay_tuning.db" 
STORAGE_URL = f"sqlite:///{DB_FILENAME}"

In [ ]:
study = optuna.create_study(
    sampler=optuna.samplers.TPESampler(),
    direction="maximize",
    study_name=STUDY_NAME,
    storage=STORAGE_URL,
    load_if_exists=True
)

In [ ]:
from optuna_dashboard import run_server

In [ ]:
def cb(study, trial):
    print("trial", trial.number, trial.state, "value:", trial.value)
    
study.optimize(objective, n_trials=50, callbacks=[cb])


In [ ]:
with run_server(STORAGE_URL):
    study.optimize(objective, n_trials=50, callbacks=[cb])

In [ ]:
for key, value in study.best_params.items():
    print(f"  - {key}: {value}")
print("="*40)

## Provide Inputs

Execute one of the following: 
- (Preferred) Assign `combined_features_train`, `combined_features_test`, and `train_data` from the feature engineering notebook (they should already be in the kernel).
- Or: Load previously saved CSV/Parquet files and provide `train_ids_sorted` (chronological train match IDs).


In [ ]:
# Example placeholders (uncomment and adapt if loading from disk):
# combined_features_train = pd.read_parquet('combined_features_train.parquet')
# combined_features_test = pd.read_parquet('combined_features_test.parquet')
# train_ids_sorted = list(pd.read_parquet('train_ids_sorted.parquet')['match_id'])

# If you ran feature_engineering.ipynb in this kernel, you likely have: 
# - combined_features_train, combined_features_test
# - train_data (list of matches in chronological order)
try:
    train_ids_sorted  # type: ignore[name-defined]
except NameError:
    # Derive sorted train IDs from train_data if available
    try:
        train_ids_sorted = [m.match_id for m in train_data]  # noqa: F821
    except Exception:
        train_ids_sorted = None

# Sanity check
assert 'combined_features_train' in globals(), 'Provide combined_features_train DataFrame'
assert 'combined_features_test' in globals(), 'Provide combined_features_test DataFrame'
if train_ids_sorted is None:
    print('WARNING: train_ids_sorted not provided; will fallback to a simple 90/10 split by row order (not time-aware).')


## Build Time-Aware Validation Split

We hold out the last 10% of the training time window as validation.


In [ ]:
df_train = combined_features_train.copy()
df_test  = combined_features_test.copy()
feature_cols = [c for c in df_train.columns if c not in ('match_id', 'radiant_win')]

if train_ids_sorted is not None:
    val_ratio = 0.10
    n_val = max(1, int(len(train_ids_sorted) * val_ratio))
    val_ids = set(train_ids_sorted[-n_val:])
    val_mask = df_train['match_id'].isin(val_ids)
else:
    # Fallback: simple row split (not time-aware)
    n = len(df_train)
    n_val = max(1, int(n * 0.10))
    val_mask = pd.Series([False]*(n-n_val) + [True]*n_val, index=df_train.index)

X_tr, y_tr = df_train.loc[~val_mask, feature_cols], df_train.loc[~val_mask, 'radiant_win'].astype(int)
X_val, y_val = df_train.loc[val_mask,  feature_cols], df_train.loc[val_mask,  'radiant_win'].astype(int)
X_te, y_te = df_test[feature_cols], df_test['radiant_win'].astype(int)

X_tr.shape, X_val.shape, X_te.shape


## LightGBM: Small, CPU-Friendly Search with Early Stopping

We try a tiny grid; early stopping halts training when validation does not improve.


In [ ]:
param_grid = [
    dict(num_leaves=31, min_data_in_leaf=200, feature_fraction=0.8, subsample=0.8, learning_rate=0.05),
    dict(num_leaves=63, min_data_in_leaf=300, feature_fraction=0.8, subsample=0.8, learning_rate=0.05),
    dict(num_leaves=63, min_data_in_leaf=500, feature_fraction=0.7, subsample=0.8, learning_rate=0.05),
    dict(num_leaves=127, min_data_in_leaf=800, feature_fraction=0.9, subsample=0.8, learning_rate=0.03),
]

best = None
for i, p in enumerate(param_grid, 1):
    model = lgb.LGBMClassifier(
        objective='binary',
        n_estimators=5000,
        early_stopping_rounds=100,
        verbosity=-1,
        random_state=42,
        n_jobs=-1,
        **p
    )
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric='binary_logloss',
        callbacks=[lgb.log_evaluation(period=100)]
    )
    y_pred = model.predict(X_val)
    acc = accuracy_score(y_val, y_pred)
    if (best is None) or (acc > best['acc']):
        best = dict(params=p, acc=acc, model=model)
    print(f'[{i}/{len(param_grid)}] val_acc={acc:.5f} params={p}')

print('Best val accuracy:', best['acc'])
print('Best params:', best['params'])


### Optional: Retrain Best Model and Score on Test


In [ ]:
best_params = best['params']
final_model = lgb.LGBMClassifier(
    objective='binary',
    n_estimators=best['model'].best_iteration_ or 1000,
    verbosity=-1,
    random_state=42,
    n_jobs=-1,
    **best_params
)
final_model.fit(df_train[feature_cols], df_train['radiant_win'].astype(int))
test_acc = accuracy_score(y_te, final_model.predict(X_te))
print('Test accuracy:', test_acc)


## PCA Exploration (Optional)

Evaluate PCA on the current feature set using the same validation split. Note: treat `match_id`/labels carefully and avoid leakage.


In [ ]:
def eval_pca(n_components_list: List[int]) -> pd.DataFrame:
    rows = []
    scaler = StandardScaler()
    Xtr_scaled = scaler.fit_transform(X_tr)
    Xval_scaled = scaler.transform(X_val)
    for n in n_components_list:
        pca = PCA(n_components=n, random_state=42)
        Xtr_p = pca.fit_transform(Xtr_scaled)
        Xval_p = pca.transform(Xval_scaled)
        model = lgb.LGBMClassifier(objective='binary', n_estimators=300, learning_rate=0.05, verbosity=-1, random_state=42)
        model.fit(Xtr_p, y_tr, eval_set=[(Xval_p, y_val)], eval_metric='binary_logloss', callbacks=[lgb.log_evaluation(period=100)])
        acc = accuracy_score(y_val, model.predict(Xval_p))
        rows.append({'n_components': n, 'val_acc': acc, 'explained_var': float(np.sum(pca.explained_variance_ratio_))})
    return pd.DataFrame(rows).sort_values('val_acc', ascending=False).reset_index(drop=True)

# Example: try a few sizes (adjust as desired)
# pca_results = eval_pca([8, 16, 25, 64])
# pca_results


## L1 Feature Selection (Optional)

Use L1-penalized logistic regression to select a subset of features; evaluate with LightGBM on the selected set.


In [ ]:
def eval_l1_selection(C_values: List[float]) -> pd.DataFrame:
    rows = []
    for C in C_values:
        selector_model = LogisticRegression(penalty='l1', solver='liblinear', C=C, random_state=42, max_iter=200)
        selector = SelectFromModel(selector_model)
        selector.fit(X_tr, y_tr)
        cols = np.array(feature_cols)[selector.get_support()]
        # Train LGBM on selected columns
        model = lgb.LGBMClassifier(objective='binary', n_estimators=600, learning_rate=0.05, verbosity=-1, random_state=42)
        model.fit(X_tr[cols], y_tr, eval_set=[(X_val[cols], y_val)], eval_metric='binary_logloss', callbacks=[lgb.log_evaluation(period=100)])
        acc = accuracy_score(y_val, model.predict(X_val[cols]))
        rows.append({'C': C, 'selected_features': int(len(cols)), 'val_acc': acc})
    return pd.DataFrame(rows).sort_values('val_acc', ascending=False).reset_index(drop=True)

# Example: try a few Cs (adjust as desired)
# l1_results = eval_l1_selection([0.001, 0.01, 0.1, 1.0])
# l1_results


# PCA

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.base import BaseEstimator, TransformerMixin

def apply_pca_transformation(
    X_train_raw, 
    X_test_raw, 
    n_components, 
    random_state=42
):
    """
    Applies scaling and PCA to reduce the dimensionality of raw feature dataframes.

    This function correctly handles the fit/transform paradigm to prevent data
    leakage from the test set. It scales the data, applies PCA, and returns
    new dataframes with the principal components and the original 'match_id'.

    Args:
        X_train_raw (pd.DataFrame): The raw, high-dimensional training data. 
                                    Must include a 'match_id' column.
        X_test_raw (pd.DataFrame): The raw, high-dimensional testing data.
                                   Must include a 'match_id' column.
        n_components (int): The number of principal components to keep.
        random_state (int): The random state for PCA reproducibility.

    Returns:
        tuple: A tuple containing two pandas DataFrames:
               - X_train_pca_df (pd.DataFrame): Transformed training data with PCA features.
               - X_test_pca_df (pd.DataFrame): Transformed testing data with PCA features.
    """
    print(f"Applying PCA transformation with n_components={n_components}...")

    # --- 1. Input Validation ---
    if 'match_id' not in X_train_raw.columns or 'match_id' not in X_test_raw.columns:
        raise ValueError("Input DataFrames must contain a 'match_id' column.")

    # --- 2. Isolate Features and IDs ---
    train_ids = X_train_raw['match_id']
    test_ids = X_test_raw['match_id']
    
    X_train_features = X_train_raw.drop(columns=['match_id'])
    X_test_features = X_test_raw.drop(columns=['match_id'])

    # --- 3. Scaling (Fit on Train, Transform Both) ---
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_features)
    X_test_scaled = scaler.transform(X_test_features)

    # --- 4. PCA (Fit on Train, Transform Both) ---
    pca = PCA(n_components=n_components, random_state=random_state)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_features)
    
    # Report explained variance
    explained_variance = pca.explained_variance_ratio_.sum()
    print(f"Explained Variance: {explained_variance:.2%}")

    # --- 5. Create Result DataFrames ---
    pca_cols = [f'PC_{i+1}' for i in range(n_components)]
    
    X_train_pca_df = pd.DataFrame(X_train_pca, columns=pca_cols, index=X_train_raw.index)
    X_test_pca_df = pd.DataFrame(X_test_pca, columns=pca_cols, index=X_test_raw.index)

    # Re-attach match_id
    X_train_pca_df['match_id'] = train_ids
    X_test_pca_df['match_id'] = test_ids

    print("PCA transformation complete.")
    return X_train_pca_df, X_test_pca_df

In [ ]:
X_train_w2v_pca, X_test_w2v_pca = apply_pca_transformation(
    X_train_raw=w2v_features_train, 
    X_test_raw=w2v_features_test,
    n_components=8
)

In [ ]:
run_experiment(
    feature_sets_train=[
        X_train_w2v_pca,
    ],
    feature_sets_test=[
        X_test_w2v_pca,
    ],
    y_test_df=test_outcome_df,
    y_train_df=train_outcome_df,
    models_dict=Models,
)